# Diagnostic Checks

Verification analyses supporting specific claims made in the report and slides.
Run these if a reviewer or audience member pushes back on a methodological point.

In [1]:
import numpy as np
import pandas as pd
import json

questions_gpt = pd.read_parquet('../data/processed/questions_gpt4o.parquet')
questions_llama = pd.read_parquet('../data/processed/questions_llama.parquet')
curves_gpt = pd.read_parquet('../data/processed/calibration_curves_gpt4o.parquet')

BINS_EDGES = np.linspace(0, 1, 21)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## 1. Confidence distribution — aggregate

Claim: the model's confidence is heavily concentrated near 1.0 (median = 1.000 for GPT).
This means reliability curves only have meaningful support in the high-confidence region.

In [2]:
print('GPT-4o-mini confidence distribution:')
print(questions_gpt['confidence'].describe())
print()
print('Percentiles:')
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f'  {p}th: {questions_gpt["confidence"].quantile(p/100):.3f}')

GPT-4o-mini confidence distribution:
count    14042.000000
mean         0.957081
std          0.123241
min          0.250000
25%          0.997522
50%          1.000000
75%          1.000000
max          1.000000
Name: confidence, dtype: float64

Percentiles:
  1th: 0.250
  5th: 0.678
  10th: 0.852
  25th: 0.998
  50th: 1.000
  75th: 1.000
  90th: 1.000
  95th: 1.000
  99th: 1.000


## 2. Confidence distribution — per subject

The support issue is subject-dependent. Math/reasoning subjects have real spread;
most other subjects are concentrated near 1.0.

In [3]:
stats = questions_gpt.groupby('subject')['confidence'].agg(
    min='min',
    p10=lambda x: x.quantile(0.10),
    p25=lambda x: x.quantile(0.25),
    median='median'
).sort_values('p10')
print('Subjects sorted by 10th percentile of confidence (most spread first):')
print(stats.to_string())

Subjects sorted by 10th percentile of confidence (most spread first):
                                          min       p10       p25    median
subject                                                                    
high_school_mathematics              0.250000  0.250000  0.250000  0.448350
college_mathematics                  0.250000  0.363181  0.726482  0.961037
abstract_algebra                     0.250000  0.499930  0.776287  0.974649
college_chemistry                    0.374468  0.595695  0.783418  0.992357
college_physics                      0.453073  0.608801  0.880766  0.989893
high_school_physics                  0.250000  0.668481  0.866375  0.999290
college_computer_science             0.250000  0.679176  0.940998  0.999967
electrical_engineering               0.404155  0.692709  0.992863  0.999999
global_facts                         0.554760  0.711646  0.851952  0.989939
business_ethics                      0.359867  0.714514  0.997052  1.000000
elementary_mathema

## 3. Are above-diagonal isotonic curves due to empty bins or genuine underconfidence?

Claim: 23/57 subjects have portions of the isotonic curve above the diagonal,
but this is entirely a data scarcity artifact — not genuine underconfidence.
Verification: check how many questions actually fall in those bins.

In [4]:
results = []
for _, row in curves_gpt.iterrows():
    subj = row['subject']
    conf_bins = np.array(row['confidence_bins'])
    iso = np.array(row['isotonic_acc'])

    above_mask = iso > conf_bins + 0.01
    if not above_mask.any():
        continue

    subj_conf = questions_gpt[questions_gpt['subject'] == subj]['confidence'].values
    bin_counts, _ = np.histogram(subj_conf, bins=BINS_EDGES)

    for b, iso_val in zip(conf_bins[above_mask], iso[above_mask]):
        bin_idx = min(int(b * 20), 19)
        results.append({
            'subject': subj,
            'bin_midpoint': round(b, 2),
            'isotonic_acc': round(iso_val, 3),
            'n_questions_in_bin': bin_counts[bin_idx],
        })

df = pd.DataFrame(results)
print(f'Subjects with any above-diagonal isotonic point: {df.subject.nunique()}/57')
print(f'Total above-diagonal bin instances: {len(df)}')
print(f'  Empty bins (0 questions):   {(df.n_questions_in_bin == 0).sum()}')
print(f'  Sparse bins (1-4 questions): {((df.n_questions_in_bin >= 1) & (df.n_questions_in_bin <= 4)).sum()}')
print(f'  Substantial (5+ questions):  {(df.n_questions_in_bin >= 5).sum()}')
print()
print('Any cases with 5+ questions (would indicate genuine underconfidence):')
print(df[df.n_questions_in_bin >= 5])

Subjects with any above-diagonal isotonic point: 23/57
Total above-diagonal bin instances: 137
  Empty bins (0 questions):   134
  Sparse bins (1-4 questions): 3
  Substantial (5+ questions):  0

Any cases with 5+ questions (would indicate genuine underconfidence):
Empty DataFrame
Columns: [subject, bin_midpoint, isotonic_acc, n_questions_in_bin]
Index: []


## 4. formal_logic kernel curve artifact

Claim: the kernel curve for formal_logic reads ~1.0 from confidence 0.03 to 0.53.
This is a Gaussian smoothing artifact — those bins have zero questions.
Verification: count questions per bin for formal_logic.

In [5]:
fl_conf = questions_gpt[questions_gpt['subject'] == 'formal_logic']['confidence'].values
print(f'formal_logic: {len(fl_conf)} total questions')
bin_counts, _ = np.histogram(fl_conf, bins=BINS_EDGES)
print('Questions per confidence bin:')
for i, c in enumerate(bin_counts):
    flag = ' <-- empty, kernel artifact' if c == 0 else ''
    print(f'  [{BINS_EDGES[i]:.2f}-{BINS_EDGES[i+1]:.2f}]: {c}{flag}')

print()
fl_row = curves_gpt[curves_gpt['subject'] == 'formal_logic'].iloc[0]
conf_bins = np.array(fl_row['confidence_bins'])
ker = np.array(fl_row['kernel_acc'])
print('Kernel curve values in low-confidence region:')
for b, k in zip(conf_bins, ker):
    if b < 0.75:
        n = bin_counts[min(int(b * 20), 19)]
        print(f'  conf={b:.2f}: kernel_acc={k:.3f}  (n={n} questions in bin)')

formal_logic: 126 total questions
Questions per confidence bin:
  [0.00-0.05]: 0 <-- empty, kernel artifact
  [0.05-0.10]: 0 <-- empty, kernel artifact
  [0.10-0.15]: 0 <-- empty, kernel artifact
  [0.15-0.20]: 0 <-- empty, kernel artifact
  [0.20-0.25]: 0 <-- empty, kernel artifact
  [0.25-0.30]: 0 <-- empty, kernel artifact
  [0.30-0.35]: 0 <-- empty, kernel artifact
  [0.35-0.40]: 0 <-- empty, kernel artifact
  [0.40-0.45]: 0 <-- empty, kernel artifact
  [0.45-0.50]: 2
  [0.50-0.55]: 0 <-- empty, kernel artifact
  [0.55-0.60]: 0 <-- empty, kernel artifact
  [0.60-0.65]: 2
  [0.65-0.70]: 1
  [0.70-0.75]: 9
  [0.75-0.80]: 1
  [0.80-0.85]: 7
  [0.85-0.90]: 5
  [0.90-0.95]: 10
  [0.95-1.00]: 89

Kernel curve values in low-confidence region:
  conf=0.03: kernel_acc=1.000  (n=0 questions in bin)
  conf=0.08: kernel_acc=1.000  (n=0 questions in bin)
  conf=0.12: kernel_acc=1.000  (n=0 questions in bin)
  conf=0.18: kernel_acc=1.000  (n=0 questions in bin)
  conf=0.23: kernel_acc=1.000  (n=

## 5. NLCS Spearman correlations

Claim: ECE subject rankings are stable when using unbinned NLCS instead of binned ECE.
Values filled into report/slides from sensitivity JSON.

In [6]:
for model in ['gpt4o', 'llama']:
    with open(f'../data/processed/sensitivity_{model}.json') as f:
        d = json.load(f)
    bins = d['ece_bins']
    min_nlcs = min(v['spearman_vs_nlcs'] for v in bins.values())
    min_base = min(v['spearman_vs_baseline'] for k, v in bins.items() if k != '20')
    print(f'{model}:')
    print(f'  Min Spearman(ECE_b, NLCS) across bin counts: {min_nlcs:.4f}')
    print(f'  Min Spearman(ECE_b, ECE_20) across bin counts: {min_base:.4f}')
    print()

gpt4o:
  Min Spearman(ECE_b, NLCS) across bin counts: 0.9025
  Min Spearman(ECE_b, ECE_20) across bin counts: 0.9970

llama:
  Min Spearman(ECE_b, NLCS) across bin counts: 0.8612
  Min Spearman(ECE_b, ECE_20) across bin counts: 0.9839

